# OpenAlex Final Dataset Analysis - Sri Lanka Only

This notebook loads the final OpenAlex dataset, enforces a strict Sri Lanka-only affiliation filter, and produces summary tables for publications, citations, fields, institutions, authors, journals, and open access.

**Strict Sri Lanka-only rule used here:** keep a work only when the known affiliation countries are exactly `LK`. Records with no detectable country code or with mixed countries such as `LK; US` are excluded from the strict dataset.

In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "processed" / "openalex"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

## 1. Select the final OpenAlex dataset

Update `DATASET_PATH` if your final dataset has a different name. Supported formats: `.csv`, `.parquet`, `.jsonl`, `.json`, `.xlsx`.

In [ ]:
candidate_paths = [
    DATA_DIR / "processed" / "openalex" / "openalex_final.csv",
    DATA_DIR / "processed" / "openalex_final.csv",
    DATA_DIR / "interim" / "openalex" / "openalex_final.csv",
    DATA_DIR / "raw" / "openalex" / "lk_works.csv",
    DATA_DIR / "raw" / "openalex" / "lk_works.jsonl",
]

existing_candidates = [path for path in candidate_paths if path.exists()]
if existing_candidates:
    DATASET_PATH = existing_candidates[0]
else:
    discovered = sorted(
        list(DATA_DIR.rglob("*openalex*")) + list(DATA_DIR.rglob("*lk_works*")),
        key=lambda path: str(path),
    )
    discovered = [path for path in discovered if path.suffix.lower() in {".csv", ".parquet", ".jsonl", ".json", ".xlsx"}]
    if not discovered:
        raise FileNotFoundError(
            "No OpenAlex dataset found under data/. Set DATASET_PATH manually to your final dataset file."
        )
    DATASET_PATH = discovered[0]

DATASET_PATH

In [ ]:
def load_dataset(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    raise ValueError(f"Unsupported dataset format: {path.suffix}")


df_raw = load_dataset(DATASET_PATH)
df_raw.columns = [str(column).strip() for column in df_raw.columns]

print(f"Loaded: {DATASET_PATH}")
print(f"Rows: {len(df_raw):,}")
print(f"Columns: {df_raw.shape[1]:,}")
df_raw.head(3)

## 2. Normalize important columns

This section handles both a flat CSV exported by `scripts/convert_openalex_jsonl_to_csv.py` and raw OpenAlex JSON/JSONL records.

In [ ]:
def first_existing_column(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lowered = {column.lower(): column for column in df.columns}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    return None


def normalize_text(value) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value).strip()


def split_multi_value(value) -> list[str]:
    text = normalize_text(value)
    if not text or text.lower() in {"nan", "none", "null"}:
        return []
    parts = re.split(r"\s*;\s*|\s*,\s*|\s*\|\s*", text)
    return [part.strip() for part in parts if part.strip()]


def unique_join(values, separator="; ") -> str:
    seen = set()
    cleaned = []
    for value in values:
        text = normalize_text(value)
        if not text or text in seen:
            continue
        seen.add(text)
        cleaned.append(text)
    return separator.join(cleaned)


def as_list(value) -> list:
    return value if isinstance(value, list) else []


def raw_country_codes(row: pd.Series) -> set[str]:
    codes: set[str] = set()

    countries_col = first_existing_column(df_raw, ["countries", "country_codes", "affiliation_countries"])
    if countries_col:
        codes.update(code.upper() for code in split_multi_value(row.get(countries_col)))

    authorships = row.get("authorships")
    if isinstance(authorships, str):
        try:
            authorships = json.loads(authorships)
        except json.JSONDecodeError:
            authorships = []

    for authorship in as_list(authorships):
        if not isinstance(authorship, dict):
            continue
        codes.update(str(code).upper() for code in as_list(authorship.get("countries")) if code)
        for institution in as_list(authorship.get("institutions")):
            if isinstance(institution, dict) and institution.get("country_code"):
                codes.add(str(institution["country_code"]).upper())

    institutions = row.get("institutions")
    if isinstance(institutions, str) and institutions.lstrip().startswith("["):
        try:
            institutions = json.loads(institutions)
        except json.JSONDecodeError:
            institutions = []

    for institution in as_list(institutions):
        if isinstance(institution, dict) and institution.get("country_code"):
            codes.add(str(institution["country_code"]).upper())

    return {code for code in codes if code and code != "NAN"}


def extract_raw_names(row: pd.Series, key: str, lk_only: bool = False) -> str:
    names = []
    authorships = row.get("authorships")
    if isinstance(authorships, str):
        try:
            authorships = json.loads(authorships)
        except json.JSONDecodeError:
            authorships = []

    for authorship in as_list(authorships):
        if not isinstance(authorship, dict):
            continue
        countries = {str(code).upper() for code in as_list(authorship.get("countries"))}
        institutions = as_list(authorship.get("institutions"))
        has_lk_institution = any(
            isinstance(inst, dict) and inst.get("country_code") == "LK" for inst in institutions
        )
        if lk_only and "LK" not in countries and not has_lk_institution:
            continue

        if key == "authors":
            author = authorship.get("author") or {}
            names.append(author.get("display_name") or authorship.get("raw_author_name"))
        elif key == "institutions":
            names.extend(inst.get("display_name") for inst in institutions if isinstance(inst, dict))

    return unique_join(names)


df = df_raw.copy()

rename_map = {
    first_existing_column(df, ["id", "openalex_id"]): "openalex_id",
    first_existing_column(df, ["display_name", "title"]): "title",
    first_existing_column(df, ["publication_year", "year"]): "publication_year",
    first_existing_column(df, ["publication_date", "date"]): "publication_date",
    first_existing_column(df, ["cited_by_count", "citations"]): "cited_by_count",
}
rename_map = {old: new for old, new in rename_map.items() if old and old != new}
df = df.rename(columns=rename_map)

if "affiliation_country_codes" not in df.columns:
    df["affiliation_country_codes"] = df_raw.apply(raw_country_codes, axis=1)
else:
    df["affiliation_country_codes"] = df["affiliation_country_codes"].apply(
        lambda value: value if isinstance(value, set) else {code.upper() for code in split_multi_value(value)}
    )

for column in ["publication_year", "cited_by_count", "author_count", "fwci"]:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

if "authors" not in df.columns and "authorships" in df_raw.columns:
    df["authors"] = df_raw.apply(lambda row: extract_raw_names(row, "authors"), axis=1)

if "sri_lankan_authors" not in df.columns and "authorships" in df_raw.columns:
    df["sri_lankan_authors"] = df_raw.apply(lambda row: extract_raw_names(row, "authors", lk_only=True), axis=1)

if "sri_lankan_institutions" not in df.columns and "authorships" in df_raw.columns:
    df["sri_lankan_institutions"] = df_raw.apply(lambda row: extract_raw_names(row, "institutions", lk_only=True), axis=1)

df[["openalex_id", "title", "publication_year", "cited_by_count", "affiliation_country_codes"]].head(5)

## 3. Apply strict Sri Lanka-only filter

In [ ]:
STRICT_COUNTRY_CODES = {"LK"}

df["is_strict_sri_lanka_only"] = df["affiliation_country_codes"].apply(lambda codes: codes == STRICT_COUNTRY_CODES)
df["has_any_sri_lanka_affiliation"] = df["affiliation_country_codes"].apply(lambda codes: "LK" in codes)

lk_only = df.loc[df["is_strict_sri_lanka_only"]].copy()
lk_mixed_or_unknown = df.loc[~df["is_strict_sri_lanka_only"]].copy()

summary_filter = pd.DataFrame(
    {
        "group": ["raw records", "strict Sri Lanka-only", "has LK but mixed countries", "no detectable LK / unknown"],
        "records": [
            len(df),
            len(lk_only),
            int(((df["has_any_sri_lanka_affiliation"]) & (~df["is_strict_sri_lanka_only"])).sum()),
            int((~df["has_any_sri_lanka_affiliation"]).sum()),
        ],
    }
)

display(summary_filter)

if lk_only.empty:
    raise ValueError(
        "Strict Sri Lanka-only dataset is empty. Check the country columns or use the converter with --sri-lanka-filter only."
    )

print("Unique country-code sets kept in strict dataset:")
display(lk_only["affiliation_country_codes"].value_counts())

In [ ]:
output_csv = OUTPUT_DIR / "openalex_sri_lanka_only.csv"
output_parquet = OUTPUT_DIR / "openalex_sri_lanka_only.parquet"

lk_export = lk_only.copy()
lk_export["affiliation_country_codes"] = lk_export["affiliation_country_codes"].apply(lambda codes: "; ".join(sorted(codes)))
lk_export.to_csv(output_csv, index=False)

try:
    lk_export.to_parquet(output_parquet, index=False)
    print(f"Saved CSV: {output_csv}")
    print(f"Saved Parquet: {output_parquet}")
except Exception as error:
    print(f"Saved CSV: {output_csv}")
    print(f"Skipped Parquet export because the engine is unavailable: {error}")

## 4. Dataset quality checks

In [ ]:
quality_rows = []
for column in ["openalex_id", "doi", "title", "publication_year", "publication_date", "cited_by_count", "sri_lankan_institutions"]:
    if column in lk_only.columns:
        quality_rows.append(
            {
                "column": column,
                "missing": int(lk_only[column].isna().sum() + (lk_only[column].astype(str).str.strip() == "").sum()),
                "missing_pct": round(float((lk_only[column].isna() | (lk_only[column].astype(str).str.strip() == "")).mean() * 100), 2),
                "unique": int(lk_only[column].nunique(dropna=True)),
            }
        )

display(pd.DataFrame(quality_rows))

if "openalex_id" in lk_only.columns:
    duplicate_openalex = lk_only[lk_only["openalex_id"].duplicated(keep=False)].sort_values("openalex_id")
    print(f"Duplicate OpenAlex IDs: {duplicate_openalex['openalex_id'].nunique():,}")
    display(duplicate_openalex[["openalex_id", "title", "publication_year"]].head(20))

## 5. Core publication and citation summary

In [ ]:
def safe_sum(series: pd.Series) -> float:
    return float(pd.to_numeric(series, errors="coerce").fillna(0).sum())


summary = {
    "publications": len(lk_only),
    "year_min": int(lk_only["publication_year"].min()) if "publication_year" in lk_only.columns and lk_only["publication_year"].notna().any() else None,
    "year_max": int(lk_only["publication_year"].max()) if "publication_year" in lk_only.columns and lk_only["publication_year"].notna().any() else None,
    "total_citations": int(safe_sum(lk_only["cited_by_count"])) if "cited_by_count" in lk_only.columns else None,
    "mean_citations": round(float(lk_only["cited_by_count"].mean()), 2) if "cited_by_count" in lk_only.columns else None,
    "median_citations": round(float(lk_only["cited_by_count"].median()), 2) if "cited_by_count" in lk_only.columns else None,
}

pd.DataFrame([summary]).T.rename(columns={0: "value"})

In [ ]:
if "publication_year" in lk_only.columns:
    yearly = (
        lk_only.dropna(subset=["publication_year"])
        .assign(publication_year=lambda data: data["publication_year"].astype(int))
        .groupby("publication_year", as_index=False)
        .agg(
            publications=("title", "size"),
            total_citations=("cited_by_count", "sum") if "cited_by_count" in lk_only.columns else ("title", "size"),
            mean_citations=("cited_by_count", "mean") if "cited_by_count" in lk_only.columns else ("title", "size"),
        )
        .sort_values("publication_year")
    )
    yearly["mean_citations"] = yearly["mean_citations"].round(2)
    display(yearly.tail(20))
else:
    print("No publication_year column found.")

## 6. Top publications by citations

In [ ]:
top_publication_columns = [
    column for column in [
        "title",
        "publication_year",
        "cited_by_count",
        "sri_lankan_authors",
        "sri_lankan_institutions",
        "source_name",
        "doi",
        "openalex_id",
    ] if column in lk_only.columns
]

if "cited_by_count" in lk_only.columns:
    display(lk_only.sort_values("cited_by_count", ascending=False)[top_publication_columns].head(25))
else:
    display(lk_only[top_publication_columns].head(25))

## 7. Type, source, field, and topic analysis

In [ ]:
def frequency_table(data: pd.DataFrame, column: str, top_n: int = 20) -> pd.DataFrame:
    if column not in data.columns:
        return pd.DataFrame({"message": [f"Column not found: {column}"]})
    table = (
        data[column]
        .replace("", np.nan)
        .dropna()
        .astype(str)
        .value_counts()
        .head(top_n)
        .rename_axis(column)
        .reset_index(name="publications")
    )
    table["share_pct"] = (table["publications"] / len(data) * 100).round(2)
    return table


for column in ["type", "source_type", "source_name", "primary_domain", "primary_field", "primary_subfield", "primary_topic"]:
    print(f"\nTop {column}")
    display(frequency_table(lk_only, column, top_n=15))

## 8. Sri Lankan institutions and authors

In [ ]:
def explode_multi_value_column(data: pd.DataFrame, column: str) -> pd.DataFrame:
    if column not in data.columns:
        return pd.DataFrame(columns=[column])
    exploded = data[[column]].copy()
    exploded[column] = exploded[column].apply(split_multi_value)
    exploded = exploded.explode(column)
    exploded[column] = exploded[column].astype(str).str.strip()
    return exploded.loc[exploded[column].ne("")]


def top_multi_value(data: pd.DataFrame, column: str, top_n: int = 25) -> pd.DataFrame:
    exploded = explode_multi_value_column(data, column)
    if exploded.empty:
        return pd.DataFrame({"message": [f"No values available for {column}"]})
    return (
        exploded[column]
        .value_counts()
        .head(top_n)
        .rename_axis(column)
        .reset_index(name="publications")
    )


display(top_multi_value(lk_only, "sri_lankan_institutions", top_n=30))
display(top_multi_value(lk_only, "sri_lankan_authors", top_n=30))

## 9. Open access and language

In [ ]:
for column in ["is_oa", "oa_status", "language"]:
    print(f"\n{column}")
    display(frequency_table(lk_only, column, top_n=20))

## 10. Optional quick charts

These cells use `matplotlib` only if it is installed. The analysis tables above work without it.

In [ ]:
try:
    import matplotlib.pyplot as plt

    if "yearly" in globals() and not yearly.empty:
        ax = yearly.plot(x="publication_year", y="publications", kind="line", marker="o", figsize=(11, 4), legend=False)
        ax.set_title("Strict Sri Lanka-only OpenAlex publications by year")
        ax.set_xlabel("Publication year")
        ax.set_ylabel("Publications")
        ax.grid(True, alpha=0.3)
        plt.show()

    if "primary_field" in lk_only.columns:
        top_fields = frequency_table(lk_only, "primary_field", top_n=12).sort_values("publications")
        ax = top_fields.plot(x="primary_field", y="publications", kind="barh", figsize=(9, 5), legend=False)
        ax.set_title("Top OpenAlex fields - Sri Lanka-only")
        ax.set_xlabel("Publications")
        ax.set_ylabel("")
        plt.show()
except ImportError:
    print("matplotlib is not installed. Install it if you want charts: pip install matplotlib")

## 11. Save analysis tables

In [ ]:
tables = {
    "filter_summary": summary_filter,
    "yearly_summary": yearly if "yearly" in globals() else pd.DataFrame(),
    "top_institutions": top_multi_value(lk_only, "sri_lankan_institutions", top_n=100),
    "top_authors": top_multi_value(lk_only, "sri_lankan_authors", top_n=100),
    "top_sources": frequency_table(lk_only, "source_name", top_n=100),
    "top_fields": frequency_table(lk_only, "primary_field", top_n=100),
    "top_topics": frequency_table(lk_only, "primary_topic", top_n=100),
}

for name, table in tables.items():
    path = OUTPUT_DIR / f"{name}.csv"
    table.to_csv(path, index=False)
    print(f"Saved {path}")